# preprocessing master driver

the separate adni, aibl, oasis2, and oasis3 driver notebooks call the same resumable pipeline. successful files are skipped. failed files remain recorded and can be tried again. set `REPLACE=True` only when test outputs need to be rebuilt after a code or parameter change.

## shared workflow

the shared pipeline runs synthstrip mask generation, mask-guided ants n4, skull stripping, ants affine registration and fixed resampling to one mni152 template, then per-volume brain-masked z-score normalization. physical gpu 2 is exposed only to synthstrip, each stage is resumable, every image receives numeric qc, and saved qc plots are limited.

In [1]:
from pathlib import Path
import json

candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'SkullStrippingProject']
PROJECT_ROOT = next(path for path in candidates if (path / 'configs' / 'pipeline_defaults.json').exists())
settings = json.loads((PROJECT_ROOT / 'configs' / 'pipeline_defaults.json').read_text())
public_settings = {key: value for key, value in settings.items() if key != 'gpu_physical_id'}
print('shared test settings')
print(public_settings)
print('gpu policy: physical gpu 2 only')

shared test settings
{'full_max_files': None, 'driver_gpu_assignment': {'ADNI': 1, 'OASIS3': 1, 'AIBL': 2}, 'aibl_conversion_test_max_files': 5, 'aibl_conversion_workers': 2, 'save_qc_plots': True, 'max_qc_plots': 3, 'replace': False, 'brain_mask': 'FreeSurfer SynthStrip v1 main model, 1 mm border', 'n4_bias_correction': True, 'n4_mask': 'SynthStrip brain mask', 'registration': 'ANTs affine', 'registration_template': 'resources/templates/MNI152_T1_1mm_brain.nii.gz', 'target_spacing_mm': [1.0, 1.0, 1.0], 'normalization': 'per-volume brain-masked Z-score with zero background'}
gpu policy: physical gpu 2 only


## running the drivers

run the aibl conversion notebook first, then run each dataset driver. for a full run change `MAX_FILES` from 3 to `None`. leave `REPLACE=False` to continue after interruption. the command line alternative is `python scripts/run_pipeline.py skullstrip --dataset DATASET --max-files 3`.